In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
rng = np.random.default_rng(42)
n_transactions = 10_000

# Base probability that each item appears in a basket
item_probabilities = {
    "Tea": 0.16,
    "Sugar": 0.18,
    "Coffee": 0.24,
    "Biscuits": 0.20,
    "Cereal": 0.18,
    "Milk": 0.27,
    "Bread": 0.30,
    "Butter": 0.13,
    "Eggs": 0.19,
    "Fruit": 0.25,
    "Chips": 0.17,
    "Yogurt": 0.12,
}

transactions = []

for _ in range(n_transactions):
    basket = {
        item
        for item, probability in item_probabilities.items()
        if rng.random() < probability
    }

    # Add planted co-occurrence patterns
    if "Tea" in basket and rng.random() < 0.75:
        basket.add("Sugar")

    if "Coffee" in basket and rng.random() < 0.65:
        basket.add("Biscuits")

    if "Cereal" in basket and rng.random() < 0.70:
        basket.add("Milk")

    if "Bread" in basket and rng.random() < 0.60:
        basket.add("Butter")

    transactions.append(basket)

# One row per basket, one column per item
basket_df = pd.DataFrame([
    {item: item in basket for item in item_probabilities}
    for basket in transactions
]).astype(bool)

print("Dataset shape:", basket_df.shape)
print("Average items per basket:", round(basket_df.sum(axis=1).mean(), 2))
print("\nMost common items:")
print(basket_df.sum().sort_values(ascending=False))

Dataset shape: (10000, 12)
Average items per basket: 2.85

Most common items:
Milk        3598
Biscuits    3164
Bread       3024
Butter      2856
Sugar       2766
Fruit       2517
Coffee      2364
Eggs        1888
Cereal      1814
Chips       1691
Tea         1618
Yogurt      1209
dtype: int64


In [3]:
basket_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Tea       10000 non-null  bool 
 1   Sugar     10000 non-null  bool 
 2   Coffee    10000 non-null  bool 
 3   Biscuits  10000 non-null  bool 
 4   Cereal    10000 non-null  bool 
 5   Milk      10000 non-null  bool 
 6   Bread     10000 non-null  bool 
 7   Butter    10000 non-null  bool 
 8   Eggs      10000 non-null  bool 
 9   Fruit     10000 non-null  bool 
 10  Chips     10000 non-null  bool 
 11  Yogurt    10000 non-null  bool 
dtypes: bool(12)
memory usage: 117.3 KB


In [4]:
basket_df.nunique()

Tea         2
Sugar       2
Coffee      2
Biscuits    2
Cereal      2
Milk        2
Bread       2
Butter      2
Eggs        2
Fruit       2
Chips       2
Yogurt      2
dtype: int64

In [5]:
basket_df.head()

,Tea,Sugar,Coffee,Biscuits,Cereal,Milk,Bread,Butter,Eggs,Fruit,Chips,Yogurt
0,False,False,False,False,True,True,False,False,True,False,False,False
1,False,False,True,True,True,True,False,False,False,False,False,False
2,True,True,False,False,False,False,False,False,True,True,False,False
3,False,False,False,False,False,False,False,False,False,True,False,True
4,False,False,False,False,False,False,True,True,False,False,False,False


In [6]:
cols = basket_df.columns

In [7]:
# for col in cols:

#     le = LabelEncoder()
#     basket_df[col] = le.fit_transform(basket_df[col])

In [8]:
basket_df.head()

,Tea,Sugar,Coffee,Biscuits,Cereal,Milk,Bread,Butter,Eggs,Fruit,Chips,Yogurt
0,False,False,False,False,True,True,False,False,True,False,False,False
1,False,False,True,True,True,True,False,False,False,False,False,False
2,True,True,False,False,False,False,False,False,True,True,False,False
3,False,False,False,False,False,False,False,False,False,True,False,True
4,False,False,False,False,False,False,True,True,False,False,False,False


In [9]:
print(basket_df.value_counts(normalize=True)*100)

Tea    Sugar  Coffee  Biscuits  Cereal  Milk   Bread  Butter  Eggs   Fruit  Chips  Yogurt
False  False  False   False     False   False  False  False   False  False  False  False     6.80
                                        True   False  False   False  False  False  False     2.54
                                        False  False  False   False  True   False  False     2.26
                                               True   True    False  False  False  False     2.21
                                               False  False   True   False  False  False     1.86
                                                                                             ... 
       True   True    False     False   False  True   False   False  False  True   False     0.01
                                                              True   True   False  False     0.01
                                                      True    True   False  False  False     0.01
                            

In [26]:
frequent_itemsets = apriori(
    basket_df,
    min_support=0.03,
    use_colnames=True,
    max_len=3
)

frequent_itemsets.sort_values(
    "support",
    ascending=False
).reset_index(drop=True)

,support,itemsets
0,0.3598,(Milk)
1,0.3164,(Biscuits)
2,0.3024,(Bread)
3,0.2856,(Butter)
4,0.2766,(Sugar)
...,...,...
96,0.0313,"(Cereal, Chips)"
97,0.0313,"(Cereal, Biscuits, Coffee)"
98,0.0305,"(Tea, Eggs)"
99,0.0303,"(Tea, Sugar, Coffee)"


In [27]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5
)

rules[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].sort_values("lift", ascending=False).reset_index(drop=True).head(20)

,antecedents,consequents,support,confidence,lift
0,"(Tea, Fruit)",(Sugar),0.0338,0.793427,2.868500
1,(Tea),(Sugar),0.1279,0.790482,2.857853
2,"(Tea, Milk)",(Sugar),0.0452,0.790210,2.856868
3,"(Tea, Coffee)",(Sugar),0.0303,0.780928,2.823311
4,"(Tea, Biscuits)",(Sugar),0.0400,0.778210,2.813486
5,"(Tea, Bread)",(Sugar),0.0365,0.754132,2.726436
6,"(Tea, Butter)",(Sugar),0.0341,0.746171,2.697652
7,"(Milk, Biscuits)",(Coffee),0.0644,0.563923,2.385461
8,"(Biscuits, Eggs)",(Coffee),0.0329,0.563356,2.383063
9,"(Butter, Coffee)",(Bread),0.0477,0.718373,2.375574


In [25]:
columns = ["antecedents", "consequents", "support", "confidence", "lift"]

biscuit_to_coffee = rules[
    rules["antecedents"].apply(lambda col: col == frozenset({"Biscuits"}))
    & rules["consequents"].apply(lambda col: col == frozenset({"Coffee"}))
    ]
coffee_to_biscuit = rules[
    rules["antecedents"].apply(lambda col: col == frozenset({"Coffee"}))
    & rules["consequents"].apply(lambda col: col == frozenset({"Biscuits"}))
    ]

display(biscuit_to_coffee[columns])
display(coffee_to_biscuit[columns])

,antecedents,consequents,support,confidence,lift
1,(Biscuits),(Coffee),0.1681,0.53129,2.247418


,antecedents,consequents,support,confidence,lift
2,(Coffee),(Biscuits),0.1681,0.711083,2.247418
